In [6]:
%load_ext autoreload
%autoreload 2
%load_ext dotenv
%dotenv

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
import csv
_=load_dotenv(find_dotenv())

# Set up OpenAI API key (replace 'your-api-key' with your actual key)
client = OpenAI()

# Step 1: Read the content of the uploaded file
file_path = '../data/Pmg_lds.md'


In [2]:
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
import csv

# Load environment variables from .env file
_ = load_dotenv(find_dotenv())

# Set up OpenAI API client
client = OpenAI()




In [3]:
# Helper function to split document into smaller chunks
def split_text(content, max_length=5000):
    """Split text into smaller chunks to fit within token limits."""
    return [content[i:i + max_length] for i in range(0, len(content), max_length)]

In [4]:
# Step 1: Read the content of the uploaded file
file_path = '../data/Pmg_lds.md'
with open(file_path, 'r', encoding='utf-8') as file:
    document_content = file.read()
    document_content = split_text(document_content)
    
print(len(document_content))


111


In [5]:
# Step 2: Define a function to generate multiple questions, answers, and quotes using GPT-4 or GPT-4o-mini
def generate_questions_and_answers(content, num_questions=3):
    # Updated prompt to ask for multiple questions and answers
    prompt = (
        f"Read the following markdown content and generate {num_questions} sensible and well-thought-out questions, "
        f"corresponding answers, and the exact quotes necessary to provide the answers from the source material. "
        f"Format each entry as:\n"
        f"Question: <Your Question>\n"
        f"Answer: <The Answer>\n"
        f"Quote: <Exact Quote from the Document>\n\n"
        f"Here is the markdown content:\n"
        f"{content}\n"
    )
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "system", "content": prompt}],
        max_tokens=3000,  # Increase max_tokens to allow for multiple questions and answers
        temperature=0.7,
        n=1
    )
    
    # Extract the content from the response
    try:
        generated_content = response.choices[0].message.content.strip()
        # print("Generated Content:\n", generated_content)  # Print generated content for debugging
    except IndexError as e:
        print("Error: No content received from the GPT model:", e)
        generated_content = ""
    
    return generated_content




In [6]:
# Step 3: Generate questions, answers, and quotes
combined_qa_statements = []
for document in document_content:
    qa_statements = generate_questions_and_answers(document, num_questions=3)
    combined_qa_statements.append(qa_statements)
qa_pairs = "\n\n".join(combined_qa_statements)


In [7]:
print(qa_pairs)

Question: What is the purpose of "Preach My Gospel" as stated by the First Presidency?
Answer: The purpose of "Preach My Gospel" is to help missionaries become better prepared, more spiritually mature, and more persuasive teachers.
Quote: "Preach My Gospel is intended to help you be a better-prepared, more spiritually mature missionary and a more persuasive teacher."

Question: How should missionaries use "Preach My Gospel" according to the introduction?
Answer: Missionaries should use "Preach My Gospel" flexibly, spending entire sessions on specific paragraphs or chapters as needed, and studying in a sequence that meets their individual needs.
Quote: "You can spend an entire study session on just a few paragraphs—or an entire chapter. You can study chapters in order or plan another sequence that better meets your needs."

Question: What is the ultimate goal of teaching the lessons outlined in "Preach My Gospel"?
Answer: The ultimate goal of teaching the lessons is to help others come 

In [14]:
def parse_output(qa_text):
    entries = []
    current_entry = {"Question": "", "Answer": "", "Quote": ""}
    
    lines = qa_text.split("\n")
    
    for line in lines:
        if line.startswith("Question:"):
            if current_entry["Question"]:  # Add the previous entry
                entries.append(current_entry)
                current_entry = {"Question": "", "Answer": "", "Quote": ""}
            current_entry["Question"] = line.replace("Question:", "").strip()
        elif line.startswith("Answer:"):
            current_entry["Answer"] = line.replace("Answer:", "").strip()
        elif line.startswith("Quote:"):
            current_entry["Quote"] = line.replace("Quote:", "").strip()
    
    # Add the last entry if it has valid data
    if current_entry["Question"] or current_entry["Answer"] or current_entry["Quote"]:
        entries.append(current_entry)
    
    return entries


In [15]:
parsed_entries = parse_output(qa_pairs)

In [16]:
import pandas as pd

df = pd.DataFrame(parsed_entries)
df.tail()

,Question,Answer,Quote
328,How can missionaries help less-active members ...,Missionaries should seek to build faith in Jes...,"""As you visit these less-active members, seek ..."
329,What insights did President Joseph F. Smith sh...,President Smith expressed that while he felt a...,"""Oh! that I could have kept that same spirit a..."
330,What is one important way to confirm Joseph Sm...,An important way to know that Joseph Smith is ...,"""Teach people that an important way to know th..."
331,How can you foster relationships with members ...,You can build relationships with members by se...,"""Build relationships with members by serving t..."
332,What should be provided to investigators after...,"After each lesson, investigators should be pro...","""After each lesson, provide investigators with..."


In [17]:
df.to_csv("./qa_pair.csv", index=False)